# 03 · Model Training, Comparison, Selection & SHAP
Runs the same pipeline as `python -m src.train` and then inspects the results.

In [ ]:
import sys, pathlib
ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd()
sys.path.insert(0, str(ROOT))

import json, pandas as pd, numpy as np, matplotlib.pyplot as plt, plotly.express as px
from src.config import DEFAULT_RAW_DATA, get_model_dir, REPORTS_DIR
from src.preprocessing import load_dataset
from src.train import run_training

## 1 · Train (3 models, randomized hyper-parameter search with SMOTENC inside CV)
Set `fast=True` for a quick smoke run (a few minutes less).

In [ ]:
meta = run_training(load_dataset(DEFAULT_RAW_DATA), fast=False)
meta["model_version"], meta["model_type"], meta["decision_threshold"]

## 2 · Comparison table
`validation (OOF)` rows drive model selection; `test` rows are the final unbiased report.

In [ ]:
table = pd.DataFrame(meta["comparison"])
table[table["Split"] == "test"][["Model", "Precision", "Recall", "F1", "ROC-AUC", "PR-AUC"]].round(4)

In [ ]:
table.round(4)

## 3 · Confusion matrices and curves

In [ ]:
from IPython.display import Image, display
for f in ["confusion_matrices.png", "pr_curves.png", "roc_curves.png"]:
    display(Image(str(REPORTS_DIR / f)))

## 4 · Business trade-off: threshold vs. precision/recall
A missed fraud usually costs more than a manual review, so the deployed threshold maximises F-beta (β=2 by default).

In [ ]:
from sklearn.metrics import precision_recall_curve
from src.predict import FraudPredictor
from src.preprocessing import clean_dataset
test = pd.read_csv(ROOT / "data" / "processed" / "test.csv")
pred = FraudPredictor.load()
proba = pd.Series([r["fraud_probability"] for r in pred.predict_dataframe(test.drop(columns=["is_fraud"]))])
p, r, t = precision_recall_curve(test["is_fraud"], proba)
fig = px.line(pd.DataFrame({"threshold": t, "precision": p[:-1], "recall": r[:-1]}), x="threshold", y=["precision", "recall"])
fig.add_vline(x=pred.decision_threshold, line_dash="dash"); fig

## 5 · SHAP explainability

In [ ]:
display(Image(str(REPORTS_DIR / "shap_global_importance.png")))

In [ ]:
from src.explainability import plot_local_explanation
suspicious = {"amount": 2400.0, "merchant_category": "electronics", "payment_channel": "online", "device_type": "desktop",
              "is_new_device": 1, "account_age_days": 5, "distance_from_home_km": 900.0, "country_mismatch": 1,
              "txn_count_24h": 9, "avg_amount_30d": 60.0, "time_since_last_txn_min": 1.0, "timestamp": "2026-03-10T03:10:00"}
res = pred.predict(suspicious)
print(res["fraud_probability"], res["risk_score"], res["risk_level"]); print(res["explanation"])
plot_local_explanation(res);

## 6 · Artifacts
Saved to `models/`: `fraud_model.pkl`, `preprocessing_pipeline.pkl`, `model_metadata.json` (+ an immutable copy in `models/versions/<version>/`).